# Rates Vol Vega Reprojection Lab

This notebook implements a research framework for reprojecting a swaption / cap-floor vega matrix onto lower-dimensional ATM volatility-surface factors.

It includes:

1. Standard PCA on absolute ATM vol changes.
2. Standard PCA on relative and log-relative ATM vol changes.
3. Weighted PCA using current book vega, with capped and smoothed weights.
4. Two-way / separable PCA using the expiry x tenor matrix structure.
5. Tucker-style projection through expiry and tenor subspaces.
6. PCA with covariance shrinkage:
   - sample covariance,
   - Ledoit-Wolf,
   - OAS,
   - manual diagonal shrinkage,
   - correlation shrinkage,
   - factor + diagonal target,
   - separable / Kronecker target.
7. Walk-forward validation over multiple history windows.
8. Hyperparameter ranking by out-of-sample projected vega PnL error.
9. Diagnostics:
   - component contribution,
   - residual PnL,
   - residual-risk heatmap,
   - component loadings,
   - stability metrics.

The notebook runs immediately using synthetic data. Replace the synthetic-data section with your real ATM surface history and current vega matrix.

## 0. Data convention

The notebook expects:

### ATM vol surface history

`vol_df`: a `pandas.DataFrame`

- index: dates,
- columns: `pd.MultiIndex([expiry, tenor])`,
- values: ATM vols in the same units used by the vega risk system.

Example columns:

```python
('1M', '1Y'), ('1M', '2Y'), ..., ('10Y', '30Y')
```

### Current vega matrix

`vega_df`: a `pandas.DataFrame`

- index: expiries,
- columns: tenors,
- values: vega sensitivity in the corresponding ATM vol bucket.

The units must be consistent with `vol_df`. For example, if vols are in normal-vol bp, vega should be PnL per 1 normal-vol bp.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.covariance import LedoitWolf, OAS

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

USE_SYNTHETIC_DATA = True

EXPIRIES = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "15Y", "20Y"]
TENORS   = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

N_DAYS_SYNTHETIC = 1250

## 1. Helpers for grids, reshaping and plotting

In [ ]:
def label_to_years(label: str) -> float:
    """
    Convert tenor/expiry labels such as '1M', '6M', '1Y', '10Y' into year fractions.
    """
    label = str(label).upper().strip()
    if label.endswith("M"):
        return float(label[:-1]) / 12.0
    if label.endswith("Y"):
        return float(label[:-1])
    return float(label)


def make_surface_columns(expiries: List[str], tenors: List[str]) -> pd.MultiIndex:
    return pd.MultiIndex.from_product([expiries, tenors], names=["expiry", "tenor"])


def ensure_multiindex_columns(vol_df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures vol_df columns are a MultiIndex of (expiry, tenor).
    If your columns are strings such as '1Yx10Y', modify this parser accordingly.
    """
    if isinstance(vol_df.columns, pd.MultiIndex):
        return vol_df

    parsed = []
    for c in vol_df.columns:
        s = str(c)
        if "x" in s.lower():
            a, b = s.lower().split("x")
            parsed.append((a.upper(), b.upper()))
        elif "_" in s:
            a, b = s.split("_", 1)
            parsed.append((a.upper(), b.upper()))
        else:
            raise ValueError(
                f"Cannot parse column {c}. Use MultiIndex columns or labels like '1Yx10Y' / '1Y_10Y'."
            )
    out = vol_df.copy()
    out.columns = pd.MultiIndex.from_tuples(parsed, names=["expiry", "tenor"])
    return out


def align_vega_to_vol_columns(vega_df: pd.DataFrame, vol_columns: pd.MultiIndex) -> np.ndarray:
    """
    Flatten the vega matrix in exactly the same order as vol_df.columns.
    """
    values = []
    missing = []
    for expiry, tenor in vol_columns:
        try:
            values.append(float(vega_df.loc[expiry, tenor]))
        except Exception:
            values.append(0.0)
            missing.append((expiry, tenor))
    if missing:
        print(f"Warning: {len(missing)} vega buckets missing from vega_df. They were filled with zero.")
    return np.asarray(values, dtype=float)


def vector_to_matrix(x: np.ndarray, expiries: List[str], tenors: List[str]) -> np.ndarray:
    return np.asarray(x).reshape(len(expiries), len(tenors))


def matrix_to_vector(x: np.ndarray) -> np.ndarray:
    return np.asarray(x).reshape(-1)


def heatmap_matrix(mat, expiries, tenors, title="", figsize=(10, 5)):
    """
    Simple heatmap helper using matplotlib only.
    """
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mat, aspect="auto")
    ax.set_xticks(np.arange(len(tenors)))
    ax.set_yticks(np.arange(len(expiries)))
    ax.set_xticklabels(tenors, rotation=45, ha="right")
    ax.set_yticklabels(expiries)
    ax.set_title(title)
    ax.set_xlabel("Tenor")
    ax.set_ylabel("Expiry")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def plot_series(s: pd.Series, title: str, ylabel: str = ""):
    fig, ax = plt.subplots(figsize=(12, 4))
    s.plot(ax=ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 2. Synthetic data generator

This section creates a realistic-enough synthetic surface to test the notebook end-to-end.

When you use real data, replace `vol_df` and `vega_df` with your desk data and keep the rest of the notebook unchanged.

In [ ]:
def generate_synthetic_rates_vol_data(
    expiries: List[str],
    tenors: List[str],
    n_days: int = 1250,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Synthetic ATM vol surfaces with low-rank expiry/tenor factors, regimes and idiosyncratic noise.
    Values are in arbitrary vol-bp-like units.
    """
    rng = np.random.default_rng(seed)

    e_years = np.array([label_to_years(e) for e in expiries])
    t_years = np.array([label_to_years(t) for t in tenors])
    E, M = len(expiries), len(tenors)

    base = (
        70
        + 35 * np.exp(-e_years[:, None] / 2.0)
        + 10 * np.exp(-((np.log(t_years[None, :]) - np.log(7.0)) ** 2) / 1.2)
        + 4 * np.log1p(t_years[None, :])
    )

    exp_level = np.ones(E)
    exp_slope = (np.log(e_years + 0.10) - np.mean(np.log(e_years + 0.10)))
    exp_slope = exp_slope / np.linalg.norm(exp_slope)
    exp_short = np.exp(-e_years / 1.0)
    exp_short = exp_short - exp_short.mean()
    exp_short = exp_short / np.linalg.norm(exp_short)
    exp_curv = -((np.log(e_years + 0.15) - np.mean(np.log(e_years + 0.15))) ** 2)
    exp_curv = exp_curv - exp_curv.mean()
    exp_curv = exp_curv / np.linalg.norm(exp_curv)
    exp_level = exp_level / np.linalg.norm(exp_level)

    ten_level = np.ones(M)
    ten_slope = (np.log(t_years) - np.mean(np.log(t_years)))
    ten_slope = ten_slope / np.linalg.norm(ten_slope)
    ten_belly = np.exp(-((np.log(t_years) - np.log(7.0)) ** 2) / 0.6)
    ten_belly = ten_belly - ten_belly.mean()
    ten_belly = ten_belly / np.linalg.norm(ten_belly)
    ten_long = np.exp(-((np.log(t_years) - np.log(30.0)) ** 2) / 0.5)
    ten_long = ten_long - ten_long.mean()
    ten_long = ten_long / np.linalg.norm(ten_long)
    ten_level = ten_level / np.linalg.norm(ten_level)

    modes = [
        (exp_level, ten_level, 1.00),
        (exp_slope, ten_level, 0.55),
        (exp_level, ten_slope, 0.45),
        (exp_short, ten_belly, 0.38),
        (exp_curv, ten_long, 0.25),
    ]

    n_modes = len(modes)
    factor_vols_low = np.array([0.90, 0.45, 0.38, 0.30, 0.22])
    factor_vols_high = np.array([2.10, 1.10, 0.90, 0.80, 0.55])
    corr = np.array([
        [1.00, 0.35, 0.30, 0.10, 0.05],
        [0.35, 1.00, 0.20, 0.25, 0.10],
        [0.30, 0.20, 1.00, 0.15, 0.20],
        [0.10, 0.25, 0.15, 1.00, 0.10],
        [0.05, 0.10, 0.20, 0.10, 1.00],
    ])

    regime = np.zeros(n_days, dtype=int)
    regime[int(0.35*n_days):int(0.50*n_days)] = 1
    regime[int(0.75*n_days):int(0.85*n_days)] = 1

    moves = np.zeros((n_days, E, M))
    for k in range(n_days):
        vols = factor_vols_high if regime[k] == 1 else factor_vols_low
        cov = np.outer(vols, vols) * corr
        f = rng.multivariate_normal(np.zeros(n_modes), cov)
        mat = np.zeros((E, M))
        for a, (em, tm, scale) in enumerate(modes):
            mat += f[a] * scale * np.outer(em, tm)

        noise_scale = 0.08 + 0.20 * np.exp(-e_years[:, None] / 0.4) + 0.04 * (t_years[None, :] > 20)
        mat += rng.normal(0.0, noise_scale, size=(E, M))

        if rng.uniform() < 0.015:
            i = rng.integers(0, min(4, E))
            j = rng.integers(max(0, M-3), M)
            mat[i, j] += rng.normal(0.0, 2.5)

        moves[k] = mat

    surfaces = np.zeros((n_days, E, M))
    surfaces[0] = base
    for k in range(1, n_days):
        surfaces[k] = np.maximum(5.0, surfaces[k-1] + moves[k])

    dates = pd.bdate_range("2019-01-01", periods=n_days)
    columns = make_surface_columns(expiries, tenors)
    vol_df = pd.DataFrame(surfaces.reshape(n_days, E*M), index=dates, columns=columns)

    vega = np.zeros((E, M))
    for i, e in enumerate(e_years):
        for j, t in enumerate(t_years):
            vega[i, j] = (
                1000 * np.exp(-((np.log(e + 0.2) - np.log(3.0)) ** 2) / 1.2)
                * np.exp(-((np.log(t) - np.log(10.0)) ** 2) / 1.0)
            )
            vega[i, j] += (
                -550 * np.exp(-((np.log(e + 0.2) - np.log(1.0)) ** 2) / 0.8)
                * np.exp(-((np.log(t) - np.log(30.0)) ** 2) / 0.5)
            )
            vega[i, j] += 250 * rng.normal()

    vega_df = pd.DataFrame(vega, index=expiries, columns=tenors)
    return vol_df, vega_df


if USE_SYNTHETIC_DATA:
    vol_df, vega_df = generate_synthetic_rates_vol_data(EXPIRIES, TENORS, N_DAYS_SYNTHETIC, RANDOM_SEED)
else:
    # Replace these lines with your real data loading.
    #
    # Example:
    # vol_df = pd.read_csv("atm_vol_history.csv", index_col=0, parse_dates=True)
    # vol_df = ensure_multiindex_columns(vol_df)
    #
    # vega_df = pd.read_csv("current_vega_matrix.csv", index_col=0)
    #
    raise NotImplementedError("Please load vol_df and vega_df here.")

vol_df = ensure_multiindex_columns(vol_df)
vol_df = vol_df.sort_index()
EXPIRIES = list(vol_df.columns.get_level_values("expiry").unique())
TENORS = list(vol_df.columns.get_level_values("tenor").unique())

vega_vec = align_vega_to_vol_columns(vega_df, vol_df.columns)

print("vol_df shape:", vol_df.shape)
print("vega_df shape:", vega_df.shape)
display(vol_df.head())
display(vega_df)

In [ ]:
latest_surface = vol_df.iloc[-1].values.reshape(len(EXPIRIES), len(TENORS))
heatmap_matrix(latest_surface, EXPIRIES, TENORS, "Latest ATM vol surface")
heatmap_matrix(vega_df.loc[EXPIRIES, TENORS].values, EXPIRIES, TENORS, "Current vega matrix")

## 3. Move transformations

Supported transformations:

1. `absolute`:
\[
X_t = \sigma_t - \sigma_{t-1}
\]

2. `relative`:
\[
X_t = rac{\sigma_t - \sigma_{t-1}}{\sigma_{t-1}}
\]

3. `log_relative`:
\[
X_t = \log(\sigma_t) - \log(\sigma_{t-1})
\]

PnL is always evaluated in absolute vol-change space:

\[
\mathrm{PnL}^{full}_t = v^\top \Delta \sigma_t
\]

In [ ]:
def compute_move_data(
    vol_df: pd.DataFrame,
    transform: str = "absolute",
    vol_floor: float = 1e-8,
) -> Dict[str, pd.DataFrame]:
    """
    Returns transformed moves, absolute moves and previous surface levels.
    """
    vol_df = vol_df.copy().astype(float)
    prev = vol_df.shift(1)
    dvol = vol_df - prev

    if transform == "absolute":
        x = dvol
    elif transform == "relative":
        x = dvol / prev.clip(lower=vol_floor)
    elif transform == "log_relative":
        x = np.log(vol_df.clip(lower=vol_floor)) - np.log(prev.clip(lower=vol_floor))
    else:
        raise ValueError(f"Unknown transform: {transform}")

    x = x.iloc[1:].replace([np.inf, -np.inf], np.nan).dropna()
    dvol = dvol.loc[x.index]
    prev = prev.loc[x.index]

    return {"X": x, "dvol": dvol, "prev": prev}


def inverse_transform_to_absolute(
    x_hat: np.ndarray,
    prev: np.ndarray,
    transform: str,
) -> np.ndarray:
    """
    Converts reconstructed transformed moves back to absolute vol changes.
    """
    if transform == "absolute":
        return x_hat
    if transform == "relative":
        return prev * x_hat
    if transform == "log_relative":
        return prev * (np.exp(x_hat) - 1.0)
    raise ValueError(f"Unknown transform: {transform}")


def linearized_component_to_absolute(
    x_component: np.ndarray,
    prev: np.ndarray,
    transform: str,
) -> np.ndarray:
    """
    Converts a component-level contribution to absolute changes.
    For log-relative, this uses first-order linearization d_sigma ~= sigma_prev * d_log_sigma.
    """
    if transform == "absolute":
        return x_component
    if transform in ("relative", "log_relative"):
        return prev * x_component
    raise ValueError(f"Unknown transform: {transform}")

## 4. Covariance estimators and shrinkage

The PCA model below supports several covariance estimators.

Important caveat: shrinkage toward a scalar identity matrix can improve eigenvalue conditioning, but it does not materially change eigenvectors if applied as \((1-lpha)C+lpha\mu I\). If your goal is cleaner loadings, diagonal, correlation, factor+diagonal or separable targets are usually more relevant.

In [ ]:
def empirical_covariance(X_centered: np.ndarray) -> np.ndarray:
    n = X_centered.shape[0]
    if n <= 1:
        raise ValueError("Need at least two observations to estimate covariance.")
    return (X_centered.T @ X_centered) / (n - 1)


def nearest_psd_symmetric(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    C = 0.5 * (C + C.T)
    vals, vecs = np.linalg.eigh(C)
    vals = np.maximum(vals, eps)
    return (vecs * vals) @ vecs.T


def separable_covariance_target(
    X_centered: np.ndarray,
    n_expiries: int,
    n_tenors: int,
) -> np.ndarray:
    """
    Estimate a Kronecker/separable covariance target for row-major flattened matrices.
    For row-major flattening, target is approximately kron(C_expiry, C_tenor).
    """
    T = X_centered.shape[0]
    mats = X_centered.reshape(T, n_expiries, n_tenors)

    C_exp = np.zeros((n_expiries, n_expiries))
    C_ten = np.zeros((n_tenors, n_tenors))

    for x in mats:
        C_exp += x @ x.T / n_tenors
        C_ten += x.T @ x / n_expiries

    C_exp /= max(T - 1, 1)
    C_ten /= max(T - 1, 1)

    target = np.kron(C_exp, C_ten)
    return nearest_psd_symmetric(target)


def estimate_covariance(
    X_centered: np.ndarray,
    method: str = "sample",
    shrinkage_alpha: float = 0.0,
    factor_rank: int = 3,
    dims: Optional[Tuple[int, int]] = None,
) -> np.ndarray:
    """
    Estimate covariance of centered observations.

    method:
        - sample
        - ledoit_wolf
        - oas
        - diagonal_shrink
        - correlation_shrink
        - factor_diag
        - separable_shrink
    """
    method = method.lower()
    X_centered = np.asarray(X_centered, dtype=float)
    n_obs, n_features = X_centered.shape

    if method == "sample":
        return empirical_covariance(X_centered)

    if method == "ledoit_wolf":
        return LedoitWolf().fit(X_centered).covariance_

    if method == "oas":
        return OAS().fit(X_centered).covariance_

    S = empirical_covariance(X_centered)
    alpha = float(shrinkage_alpha)
    alpha = min(max(alpha, 0.0), 1.0)

    if method == "diagonal_shrink":
        target = np.diag(np.diag(S))
        return nearest_psd_symmetric((1 - alpha) * S + alpha * target)

    if method == "correlation_shrink":
        d = np.sqrt(np.maximum(np.diag(S), 1e-16))
        inv_d = 1.0 / d
        R = (S * inv_d[:, None]) * inv_d[None, :]
        R_target = (1 - alpha) * R + alpha * np.eye(n_features)
        C = (R_target * d[:, None]) * d[None, :]
        return nearest_psd_symmetric(C)

    if method == "factor_diag":
        vals, vecs = np.linalg.eigh(S)
        order = np.argsort(vals)[::-1]
        vals, vecs = vals[order], vecs[:, order]
        r = min(int(factor_rank), n_features, max(n_obs - 1, 1))
        vals_r = np.maximum(vals[:r], 0.0)
        U_r = vecs[:, :r]
        factor_cov = (U_r * vals_r) @ U_r.T
        resid = S - factor_cov
        diag_resid = np.diag(np.maximum(np.diag(resid), 1e-12))
        target = nearest_psd_symmetric(factor_cov + diag_resid)
        return nearest_psd_symmetric((1 - alpha) * S + alpha * target)

    if method == "separable_shrink":
        if dims is None:
            raise ValueError("dims=(n_expiries, n_tenors) required for separable_shrink.")
        target = separable_covariance_target(X_centered, dims[0], dims[1])
        tr_s = np.trace(S)
        tr_t = np.trace(target)
        if tr_t > 1e-16:
            target = target * (tr_s / tr_t)
        return nearest_psd_symmetric((1 - alpha) * S + alpha * target)

    raise ValueError(f"Unknown covariance method: {method}")

## 5. PCA model implementation

Weighted PCA is implemented by running PCA on:

\[
Y_t = (X_t - \mu) \odot w
\]

and converting reconstructed components back to original transformed units by dividing by \(w\).

This corresponds to PCA under a weighted metric. Large weights make the model focus more on those buckets.

In [ ]:
@dataclass
class PCAModel:
    transform: str
    n_components: int
    mean_: np.ndarray
    weights_: np.ndarray
    components_weighted_: np.ndarray
    eigenvalues_: np.ndarray
    covariance_method: str
    metadata: Dict[str, Any]

    def scores(self, X: np.ndarray) -> np.ndarray:
        Y = (np.asarray(X) - self.mean_) * self.weights_
        return Y @ self.components_weighted_.T

    def reconstruct_transformed(self, X: np.ndarray) -> np.ndarray:
        S = self.scores(X)
        Y_hat = S @ self.components_weighted_
        X_hat = Y_hat / self.weights_ + self.mean_
        return X_hat

    def component_loading_original_units(self, component_idx: int) -> np.ndarray:
        """
        Loading vector in original transformed units for one unit of weighted-space score.
        """
        return self.components_weighted_[component_idx] / self.weights_


def fit_pca_model(
    X_train: np.ndarray,
    transform: str = "absolute",
    n_components: int = 3,
    weights: Optional[np.ndarray] = None,
    covariance_method: str = "sample",
    shrinkage_alpha: float = 0.0,
    factor_rank: int = 3,
    dims: Optional[Tuple[int, int]] = None,
    demean: bool = True,
    metadata: Optional[Dict[str, Any]] = None,
) -> PCAModel:
    X_train = np.asarray(X_train, dtype=float)
    n_obs, n_features = X_train.shape

    if weights is None:
        weights = np.ones(n_features)
    weights = np.asarray(weights, dtype=float)
    weights = np.maximum(weights, 1e-12)

    mean_ = X_train.mean(axis=0) if demean else np.zeros(n_features)
    Y_centered = (X_train - mean_) * weights

    C = estimate_covariance(
        Y_centered,
        method=covariance_method,
        shrinkage_alpha=shrinkage_alpha,
        factor_rank=factor_rank,
        dims=dims,
    )

    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    k = min(int(n_components), n_features, vecs.shape[1])
    components = vecs[:, :k].T

    return PCAModel(
        transform=transform,
        n_components=k,
        mean_=mean_,
        weights_=weights,
        components_weighted_=components,
        eigenvalues_=vals[:k],
        covariance_method=covariance_method,
        metadata=metadata or {},
    )


def make_vega_weights(
    vega_vec: np.ndarray,
    power: float = 0.5,
    w_min: float = 0.10,
    w_max: float = 5.0,
    epsilon: float = 1e-8,
) -> np.ndarray:
    """
    Robust current-vega weights.

    power = 0.0 gives all ones.
    power = 0.25 or 0.5 is usually safer than power = 1.0.
    """
    v = np.abs(np.asarray(vega_vec, dtype=float))
    positive = v[v > epsilon]
    scale = np.median(positive) if len(positive) else 1.0
    raw = ((v + epsilon) / (scale + epsilon)) ** power
    return np.clip(raw, w_min, w_max)


def pca_component_pnl_contributions(
    model: PCAModel,
    X: np.ndarray,
    prev: np.ndarray,
    vega_vec: np.ndarray,
) -> pd.DataFrame:
    """
    Component-level linearized PnL contributions.
    For log-relative, the component-to-absolute conversion is first-order.
    """
    X = np.asarray(X)
    prev = np.asarray(prev)
    scores = model.scores(X)
    out = {}

    for j in range(model.n_components):
        loading = model.component_loading_original_units(j)
        comp_transformed = scores[:, [j]] * loading[None, :]
        comp_abs = linearized_component_to_absolute(comp_transformed, prev, model.transform)
        out[f"PC{j+1}"] = comp_abs @ vega_vec

    return pd.DataFrame(out)

## 6. Two-way / separable PCA implementation

This treats each daily move as a matrix:

\[
X_t \in \mathbb{R}^{E 	imes M}
\]

and estimates expiry factors \(A\) and tenor factors \(B\). Projection is:

\[
\widehat{X}_t = A A^	op X_t B B^	op
\]

This is also the practical out-of-sample projection used by a Tucker-style model with fixed expiry and tenor subspaces.

In [ ]:
@dataclass
class SeparablePCAModel:
    transform: str
    r_expiry: int
    r_tenor: int
    mean_matrix_: np.ndarray
    A_: np.ndarray
    B_: np.ndarray
    eig_expiry_: np.ndarray
    eig_tenor_: np.ndarray
    metadata: Dict[str, Any]

    def reconstruct_transformed(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        T = X.shape[0]
        E, M = self.mean_matrix_.shape
        mats = X.reshape(T, E, M)
        recon = np.zeros_like(mats)
        for t in range(T):
            xc = mats[t] - self.mean_matrix_
            recon[t] = self.A_ @ (self.A_.T @ xc @ self.B_) @ self.B_.T + self.mean_matrix_
        return recon.reshape(T, E * M)

    def scores(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        T = X.shape[0]
        E, M = self.mean_matrix_.shape
        mats = X.reshape(T, E, M)
        scores = np.zeros((T, self.r_expiry, self.r_tenor))
        for t in range(T):
            xc = mats[t] - self.mean_matrix_
            scores[t] = self.A_.T @ xc @ self.B_
        return scores


def fit_separable_pca_model(
    X_train: np.ndarray,
    n_expiries: int,
    n_tenors: int,
    transform: str = "absolute",
    r_expiry: int = 3,
    r_tenor: int = 3,
    metadata: Optional[Dict[str, Any]] = None,
) -> SeparablePCAModel:
    X_train = np.asarray(X_train, dtype=float)
    T = X_train.shape[0]
    mats = X_train.reshape(T, n_expiries, n_tenors)
    mean_matrix = mats.mean(axis=0)
    centered = mats - mean_matrix

    C_exp = np.zeros((n_expiries, n_expiries))
    C_ten = np.zeros((n_tenors, n_tenors))

    for x in centered:
        C_exp += x @ x.T / n_tenors
        C_ten += x.T @ x / n_expiries
    C_exp /= max(T - 1, 1)
    C_ten /= max(T - 1, 1)

    vals_e, vecs_e = np.linalg.eigh(C_exp)
    vals_m, vecs_m = np.linalg.eigh(C_ten)

    order_e = np.argsort(vals_e)[::-1]
    order_m = np.argsort(vals_m)[::-1]

    vals_e, vecs_e = vals_e[order_e], vecs_e[:, order_e]
    vals_m, vecs_m = vals_m[order_m], vecs_m[:, order_m]

    rE = min(int(r_expiry), n_expiries)
    rM = min(int(r_tenor), n_tenors)

    return SeparablePCAModel(
        transform=transform,
        r_expiry=rE,
        r_tenor=rM,
        mean_matrix_=mean_matrix,
        A_=vecs_e[:, :rE],
        B_=vecs_m[:, :rM],
        eig_expiry_=vals_e[:rE],
        eig_tenor_=vals_m[:rM],
        metadata=metadata or {},
    )


def separable_component_pnl_contributions(
    model: SeparablePCAModel,
    X: np.ndarray,
    prev: np.ndarray,
    vega_matrix: np.ndarray,
) -> pd.DataFrame:
    """
    Linearized PnL contribution of each expiry-factor x tenor-factor component.
    """
    scores = model.scores(X)
    T = scores.shape[0]
    prev = np.asarray(prev)
    E, M = model.mean_matrix_.shape

    out = {}
    for i in range(model.r_expiry):
        for j in range(model.r_tenor):
            loading = np.outer(model.A_[:, i], model.B_[:, j])
            comp = scores[:, i, j].reshape(T, 1, 1) * loading[None, :, :]
            comp_flat = comp.reshape(T, E*M)
            comp_abs = linearized_component_to_absolute(comp_flat, prev, model.transform)
            out[f"E{i+1}xT{j+1}"] = comp_abs @ vega_matrix.reshape(-1)
    return pd.DataFrame(out)

## 7. Optional TensorLy check for Tucker decomposition

The operational out-of-sample projection used in this notebook is the expiry/tenor-subspace projection above. If TensorLy is installed, the following helper can also compute an in-sample Tucker decomposition of the training tensor for diagnostics.

This is optional. The notebook does not depend on TensorLy.

In [ ]:
def optional_tucker_diagnostics(X_train: np.ndarray, n_expiries: int, n_tenors: int, ranks=(5, 3, 3)):
    """
    Optional in-sample Tucker diagnostics if tensorly is installed.

    ranks = (time_rank, expiry_rank, tenor_rank)
    """
    try:
        import tensorly as tl
        from tensorly.decomposition import tucker
    except Exception:
        print("TensorLy is not installed. Skipping optional Tucker diagnostics.")
        print("Install with: pip install tensorly")
        return None

    T = X_train.shape[0]
    tensor = X_train.reshape(T, n_expiries, n_tenors)
    tensor = tensor - tensor.mean(axis=0, keepdims=True)

    try:
        core, factors = tucker(tensor, rank=ranks, init="svd", tol=1e-6, n_iter_max=200)
    except TypeError:
        core, factors = tucker(tensor, ranks=ranks, init="svd", tol=1e-6, n_iter_max=200)

    recon = tl.tucker_to_tensor((core, factors))
    err = np.sqrt(np.mean((tensor - recon) ** 2))
    print(f"Tucker in-sample tensor RMSE: {err:.6g}")
    return {"core": core, "factors": factors, "reconstruction_rmse": err}

## 8. Walk-forward validation engine

The model is recalibrated on a rolling history window and tested on the following out-of-sample period.

The main objective is projected vega PnL error:

\[
arepsilon_t = v^	op \Delta \sigma_t - v^	op \widehat{\Delta \sigma}_t
\]

In [ ]:
@dataclass
class ModelSpec:
    name: str
    family: str
    transform: str = "absolute"
    n_components: int = 3
    covariance_method: str = "sample"
    shrinkage_alpha: float = 0.0
    factor_rank: int = 3
    vega_weight_power: float = 0.0
    weight_min: float = 0.10
    weight_max: float = 5.0
    r_expiry: int = 3
    r_tenor: int = 3


def pnl_metrics(full_pnl: np.ndarray, proj_pnl: np.ndarray) -> Dict[str, float]:
    full_pnl = np.asarray(full_pnl, dtype=float)
    proj_pnl = np.asarray(proj_pnl, dtype=float)
    err = full_pnl - proj_pnl

    var_full = np.var(full_pnl, ddof=1) if len(full_pnl) > 1 else np.nan
    var_err = np.var(err, ddof=1) if len(err) > 1 else np.nan
    explained = 1.0 - var_err / var_full if var_full and var_full > 1e-16 else np.nan

    corr = np.corrcoef(full_pnl, proj_pnl)[0, 1] if len(full_pnl) > 2 and np.std(proj_pnl) > 1e-16 else np.nan

    return {
        "n_test_obs": len(err),
        "full_pnl_std": float(np.std(full_pnl, ddof=1)) if len(full_pnl) > 1 else np.nan,
        "proj_pnl_std": float(np.std(proj_pnl, ddof=1)) if len(proj_pnl) > 1 else np.nan,
        "error_mean": float(np.mean(err)),
        "error_rmse": float(np.sqrt(np.mean(err ** 2))),
        "error_mae": float(np.mean(np.abs(err))),
        "error_p95_abs": float(np.quantile(np.abs(err), 0.95)),
        "error_max_abs": float(np.max(np.abs(err))),
        "explained_pnl_variance": float(explained),
        "full_vs_projected_corr": float(corr),
    }


def fit_model_from_spec(
    spec: ModelSpec,
    X_train: np.ndarray,
    vega_vec: np.ndarray,
    n_expiries: int,
    n_tenors: int,
):
    if spec.family == "pca":
        weights = make_vega_weights(
            vega_vec,
            power=spec.vega_weight_power,
            w_min=spec.weight_min,
            w_max=spec.weight_max,
        )
        return fit_pca_model(
            X_train,
            transform=spec.transform,
            n_components=spec.n_components,
            weights=weights,
            covariance_method=spec.covariance_method,
            shrinkage_alpha=spec.shrinkage_alpha,
            factor_rank=spec.factor_rank,
            dims=(n_expiries, n_tenors),
            metadata=asdict(spec),
        )
    elif spec.family == "separable":
        return fit_separable_pca_model(
            X_train,
            n_expiries=n_expiries,
            n_tenors=n_tenors,
            transform=spec.transform,
            r_expiry=spec.r_expiry,
            r_tenor=spec.r_tenor,
            metadata=asdict(spec),
        )
    else:
        raise ValueError(f"Unknown model family: {spec.family}")


def walk_forward_evaluate_spec(
    spec: ModelSpec,
    vol_df: pd.DataFrame,
    vega_vec: np.ndarray,
    window: int = 250,
    test_horizon: int = 20,
    step: int = 20,
    min_train_obs: Optional[int] = None,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Walk-forward evaluation for one model spec and one history window length.
    """
    min_train_obs = min_train_obs or window

    move_data = compute_move_data(vol_df, transform=spec.transform)
    X_df = move_data["X"]
    dvol_df = move_data["dvol"]
    prev_df = move_data["prev"]

    X_all = X_df.values
    dvol_all = dvol_df.values
    prev_all = prev_df.values
    dates = X_df.index

    n_obs, n_features = X_all.shape
    n_expiries = len(vol_df.columns.get_level_values("expiry").unique())
    n_tenors = len(vol_df.columns.get_level_values("tenor").unique())

    all_rows = []
    start = min_train_obs

    while start < n_obs:
        train_start = max(0, start - window)
        train_end = start
        test_start = start
        test_end = min(start + test_horizon, n_obs)

        if train_end - train_start < min_train_obs or test_end <= test_start:
            start += step
            continue

        X_train = X_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        dvol_test = dvol_all[test_start:test_end]
        prev_test = prev_all[test_start:test_end]

        try:
            model = fit_model_from_spec(spec, X_train, vega_vec, n_expiries, n_tenors)
            X_hat_test = model.reconstruct_transformed(X_test)
            dvol_hat_test = inverse_transform_to_absolute(X_hat_test, prev_test, spec.transform)

            full_pnl = dvol_test @ vega_vec
            proj_pnl = dvol_hat_test @ vega_vec
            err = full_pnl - proj_pnl

            for i, dt in enumerate(dates[test_start:test_end]):
                all_rows.append({
                    "date": dt,
                    "train_start": dates[train_start],
                    "train_end": dates[train_end - 1],
                    "model": spec.name,
                    "window": window,
                    "transform": spec.transform,
                    "family": spec.family,
                    "full_pnl": full_pnl[i],
                    "projected_pnl": proj_pnl[i],
                    "error": err[i],
                })
        except Exception as exc:
            print(f"Model failed: {spec.name}, window={window}, split_start={dates[start]} -> {exc}")

        start += step

    pred_df = pd.DataFrame(all_rows)
    if len(pred_df) == 0:
        summary = {"model": spec.name, "window": window, "status": "failed_or_no_predictions"}
        return summary, pred_df

    m = pnl_metrics(pred_df["full_pnl"].values, pred_df["projected_pnl"].values)
    summary = {
        "model": spec.name,
        "family": spec.family,
        "transform": spec.transform,
        "window": window,
        **m,
        **{f"spec_{k}": v for k, v in asdict(spec).items()},
    }
    return summary, pred_df


def walk_forward_grid(
    specs: List[ModelSpec],
    vol_df: pd.DataFrame,
    vega_vec: np.ndarray,
    windows: List[int],
    test_horizon: int = 20,
    step: int = 20,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    summaries = []
    predictions = []

    total = len(specs) * len(windows)
    count = 0

    for spec in specs:
        for w in windows:
            count += 1
            print(f"[{count}/{total}] Evaluating {spec.name}, window={w}")
            summary, pred = walk_forward_evaluate_spec(
                spec,
                vol_df,
                vega_vec,
                window=w,
                test_horizon=test_horizon,
                step=step,
            )
            summaries.append(summary)
            if len(pred):
                predictions.append(pred)

    summary_df = pd.DataFrame(summaries)
    pred_df = pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame()
    return summary_df, pred_df

## 9. Define candidate models

This grid is intentionally moderate so the notebook runs quickly. For the real project, expand the grid after confirming the data pipeline.

In [ ]:
candidate_specs = []

for k in [1, 2, 3, 5, 8, 10]:
    candidate_specs.append(ModelSpec(
        name=f"Abs PCA sample k={k}",
        family="pca",
        transform="absolute",
        n_components=k,
        covariance_method="sample",
    ))

for k in [2, 3, 5, 8]:
    candidate_specs.append(ModelSpec(
        name=f"Rel PCA sample k={k}",
        family="pca",
        transform="relative",
        n_components=k,
        covariance_method="sample",
    ))

for k in [2, 3, 5, 8]:
    candidate_specs.append(ModelSpec(
        name=f"LogRel PCA sample k={k}",
        family="pca",
        transform="log_relative",
        n_components=k,
        covariance_method="sample",
    ))

for p in [0.25, 0.5, 1.0]:
    for k in [3, 5, 8]:
        candidate_specs.append(ModelSpec(
            name=f"Abs weighted PCA p={p} k={k}",
            family="pca",
            transform="absolute",
            n_components=k,
            covariance_method="sample",
            vega_weight_power=p,
            weight_max=5.0,
        ))

for method in ["ledoit_wolf", "oas"]:
    for k in [3, 5, 8]:
        candidate_specs.append(ModelSpec(
            name=f"Abs PCA {method} k={k}",
            family="pca",
            transform="absolute",
            n_components=k,
            covariance_method=method,
        ))

for method in ["diagonal_shrink", "correlation_shrink", "factor_diag", "separable_shrink"]:
    for alpha in [0.2, 0.5, 0.8]:
        for k in [3, 5, 8]:
            candidate_specs.append(ModelSpec(
                name=f"Abs PCA {method} alpha={alpha} k={k}",
                family="pca",
                transform="absolute",
                n_components=k,
                covariance_method=method,
                shrinkage_alpha=alpha,
                factor_rank=3,
            ))

for rE, rM in [(1, 1), (2, 2), (3, 2), (2, 3), (3, 3), (4, 3), (3, 4)]:
    candidate_specs.append(ModelSpec(
        name=f"Separable PCA rE={rE} rT={rM}",
        family="separable",
        transform="absolute",
        r_expiry=rE,
        r_tenor=rM,
    ))

print(f"Number of candidate specs: {len(candidate_specs)}")
candidate_specs[:5]

## 10. Run walk-forward validation

Default windows:

- 3M approximately 60 business days,
- 6M approximately 125 business days,
- 1Y approximately 250 business days,
- 2Y approximately 500 business days.

For a large real grid, start with fewer specs and fewer windows, then expand.

In [ ]:
WINDOWS = [60, 125, 250, 500]
TEST_HORIZON = 20
STEP = 20

summary_df, predictions_df = walk_forward_grid(
    candidate_specs,
    vol_df,
    vega_vec,
    windows=WINDOWS,
    test_horizon=TEST_HORIZON,
    step=STEP,
)

summary_df = summary_df.sort_values(["error_rmse", "error_p95_abs"], ascending=True)
display(summary_df.head(20))

In [ ]:
summary_df.to_csv("pca_reprojection_hyperparameter_results.csv", index=False)
predictions_df.to_csv("pca_reprojection_oos_predictions.csv", index=False)

print("Saved:")
print(" - pca_reprojection_hyperparameter_results.csv")
print(" - pca_reprojection_oos_predictions.csv")

## 11. Analyze best hyperparameters and history windows

In [ ]:
cols = [
    "model", "window", "family", "transform",
    "error_rmse", "error_mae", "error_p95_abs", "error_max_abs",
    "explained_pnl_variance", "full_vs_projected_corr"
]

best_by_window = (
    summary_df
    .sort_values("error_rmse")
    .groupby("window")
    .head(5)
    [cols]
)

display(best_by_window)

In [ ]:
window_perf = (
    summary_df
    .groupby("window")
    .agg(
        median_rmse=("error_rmse", "median"),
        best_rmse=("error_rmse", "min"),
        median_explained_pnl=("explained_pnl_variance", "median"),
        best_explained_pnl=("explained_pnl_variance", "max"),
    )
    .reset_index()
    .sort_values("window")
)

display(window_perf)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(window_perf["window"], window_perf["best_rmse"], marker="o", label="Best RMSE")
ax.plot(window_perf["window"], window_perf["median_rmse"], marker="o", label="Median RMSE")
ax.set_title("OOS PnL error by history window")
ax.set_xlabel("Calibration window length, business days")
ax.set_ylabel("OOS RMSE")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
group_perf = (
    summary_df
    .groupby(["family", "transform", "spec_covariance_method"], dropna=False)
    .agg(
        best_rmse=("error_rmse", "min"),
        median_rmse=("error_rmse", "median"),
        best_explained_pnl=("explained_pnl_variance", "max"),
    )
    .reset_index()
    .sort_values("best_rmse")
)

display(group_perf.head(30))

## 12. Fit the best model on the latest window and inspect diagnostics

In [ ]:
best_row = summary_df.iloc[0]
best_model_name = best_row["model"]
best_window = int(best_row["window"])

best_spec = next(s for s in candidate_specs if s.name == best_model_name)

print("Best model:")
print(best_model_name)
print("Best window:", best_window)
display(best_row[cols])

move_data = compute_move_data(vol_df, transform=best_spec.transform)
X_df = move_data["X"]
dvol_df = move_data["dvol"]
prev_df = move_data["prev"]

X_all = X_df.values
dvol_all = dvol_df.values
prev_all = prev_df.values

X_train_latest = X_all[-best_window:]
model_latest = fit_model_from_spec(
    best_spec,
    X_train_latest,
    vega_vec,
    len(EXPIRIES),
    len(TENORS),
)

X_hat = model_latest.reconstruct_transformed(X_train_latest)
dvol_hat = inverse_transform_to_absolute(X_hat, prev_all[-best_window:], best_spec.transform)

full_pnl = dvol_all[-best_window:] @ vega_vec
proj_pnl = dvol_hat @ vega_vec
err = full_pnl - proj_pnl

diagnostic_df = pd.DataFrame({
    "full_pnl": full_pnl,
    "projected_pnl": proj_pnl,
    "error": err,
}, index=X_df.index[-best_window:])

display(pd.Series(pnl_metrics(full_pnl, proj_pnl)))
display(diagnostic_df.tail())

In [ ]:
plot_series(diagnostic_df["full_pnl"], "Full vega PnL over latest calibration window", "PnL")
plot_series(diagnostic_df["projected_pnl"], "Projected vega PnL over latest calibration window", "PnL")
plot_series(diagnostic_df["error"], "Projection error over latest calibration window", "PnL")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(diagnostic_df["full_pnl"], diagnostic_df["projected_pnl"], alpha=0.6)
lims = [
    min(diagnostic_df["full_pnl"].min(), diagnostic_df["projected_pnl"].min()),
    max(diagnostic_df["full_pnl"].max(), diagnostic_df["projected_pnl"].max()),
]
ax.plot(lims, lims, linestyle="--")
ax.set_title("Full vs projected vega PnL")
ax.set_xlabel("Full vega PnL")
ax.set_ylabel("Projected vega PnL")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Component loadings and component PnL contributions

In [ ]:
def plot_pca_loadings(model: PCAModel, expiries, tenors, max_components=6):
    k = min(model.n_components, max_components)
    for j in range(k):
        loading = model.component_loading_original_units(j).reshape(len(expiries), len(tenors))
        heatmap_matrix(loading, expiries, tenors, title=f"PCA loading PC{j+1} in transformed units")


def plot_separable_loadings(model: SeparablePCAModel, expiries, tenors, max_components=6):
    count = 0
    for i in range(model.r_expiry):
        for j in range(model.r_tenor):
            if count >= max_components:
                return
            loading = np.outer(model.A_[:, i], model.B_[:, j])
            heatmap_matrix(loading, expiries, tenors, title=f"Separable loading E{i+1} x T{j+1}")
            count += 1


if isinstance(model_latest, PCAModel):
    plot_pca_loadings(model_latest, EXPIRIES, TENORS, max_components=6)
elif isinstance(model_latest, SeparablePCAModel):
    plot_separable_loadings(model_latest, EXPIRIES, TENORS, max_components=6)

In [ ]:
if isinstance(model_latest, PCAModel):
    contrib_df = pca_component_pnl_contributions(
        model_latest,
        X_train_latest,
        prev_all[-best_window:],
        vega_vec,
    )
elif isinstance(model_latest, SeparablePCAModel):
    contrib_df = separable_component_pnl_contributions(
        model_latest,
        X_train_latest,
        prev_all[-best_window:],
        vega_df.loc[EXPIRIES, TENORS].values,
    )
else:
    contrib_df = pd.DataFrame()

contrib_df.index = X_df.index[-best_window:]

component_stats = pd.DataFrame({
    "mean_contribution": contrib_df.mean(),
    "std_contribution": contrib_df.std(),
    "avg_abs_contribution": contrib_df.abs().mean(),
    "max_abs_contribution": contrib_df.abs().max(),
}).sort_values("avg_abs_contribution", ascending=False)

display(component_stats.head(20))

top_components = component_stats.head(6).index.tolist()
fig, ax = plt.subplots(figsize=(12, 5))
contrib_df[top_components].sum(axis=1).plot(ax=ax, label="Top-components total contribution")
diagnostic_df["projected_pnl"].plot(ax=ax, label="Projected PnL", alpha=0.7)
ax.set_title("Top component contribution versus projected PnL")
ax.set_ylabel("PnL")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

display(contrib_df[top_components].tail())

## 14. Residual-risk heatmap

Bucket-level residual risk:

\[
\mathrm{ResidualRisk}_{i,j}
=
|V_{i,j}|
\cdot
\mathrm{Std}
\left(
\Delta \sigma_{t,i,j} - \widehat{\Delta \sigma}_{t,i,j}
\right)
\]

This highlights buckets where the projection is dangerous because the model does not explain the bucket well and the book has material vega there.

In [ ]:
residual_dvol = dvol_all[-best_window:] - dvol_hat

bucket_residual_std = residual_dvol.std(axis=0, ddof=1)
bucket_residual_risk = np.abs(vega_vec) * bucket_residual_std
bucket_noise_ratio = (
    residual_dvol.var(axis=0, ddof=1)
    / np.maximum(dvol_all[-best_window:].var(axis=0, ddof=1), 1e-16)
)

heatmap_matrix(
    bucket_residual_risk.reshape(len(EXPIRIES), len(TENORS)),
    EXPIRIES,
    TENORS,
    title="Residual risk = |vega| x residual vol std",
)

heatmap_matrix(
    bucket_noise_ratio.reshape(len(EXPIRIES), len(TENORS)),
    EXPIRIES,
    TENORS,
    title="Noise ratio = residual variance / original variance",
)

residual_risk_table = pd.DataFrame({
    "expiry": vol_df.columns.get_level_values("expiry"),
    "tenor": vol_df.columns.get_level_values("tenor"),
    "vega": vega_vec,
    "residual_vol_std": bucket_residual_std,
    "residual_risk": bucket_residual_risk,
    "noise_ratio": bucket_noise_ratio,
}).sort_values("residual_risk", ascending=False)

display(residual_risk_table.head(20))

## 15. Component stability across rolling windows

For PCA models, we compare the subspace spanned by the first \(k\) components across consecutive rolling windows.

For two-way/separable PCA, we compare expiry and tenor subspaces separately.

In [ ]:
def projection_matrix(U: np.ndarray) -> np.ndarray:
    return U @ U.T


def pca_subspace_distance(model_a: PCAModel, model_b: PCAModel) -> float:
    Ua = model_a.components_weighted_.T
    Ub = model_b.components_weighted_.T
    k = min(Ua.shape[1], Ub.shape[1])
    Pa = projection_matrix(Ua[:, :k])
    Pb = projection_matrix(Ub[:, :k])
    return float(np.linalg.norm(Pa - Pb, ord="fro"))


def separable_subspace_distance(model_a: SeparablePCAModel, model_b: SeparablePCAModel) -> Dict[str, float]:
    kE = min(model_a.A_.shape[1], model_b.A_.shape[1])
    kM = min(model_a.B_.shape[1], model_b.B_.shape[1])
    PEa = projection_matrix(model_a.A_[:, :kE])
    PEb = projection_matrix(model_b.A_[:, :kE])
    PMa = projection_matrix(model_a.B_[:, :kM])
    PMb = projection_matrix(model_b.B_[:, :kM])
    return {
        "expiry_subspace_distance": float(np.linalg.norm(PEa - PEb, ord="fro")),
        "tenor_subspace_distance": float(np.linalg.norm(PMa - PMb, ord="fro")),
    }


def rolling_stability_analysis(
    spec: ModelSpec,
    vol_df: pd.DataFrame,
    vega_vec: np.ndarray,
    window: int,
    step: int = 20,
) -> pd.DataFrame:
    move_data = compute_move_data(vol_df, transform=spec.transform)
    X_all = move_data["X"].values
    dates = move_data["X"].index
    n_obs = len(X_all)
    n_expiries = len(vol_df.columns.get_level_values("expiry").unique())
    n_tenors = len(vol_df.columns.get_level_values("tenor").unique())

    models = []
    model_dates = []
    for end in range(window, n_obs + 1, step):
        X_train = X_all[end-window:end]
        try:
            model = fit_model_from_spec(spec, X_train, vega_vec, n_expiries, n_tenors)
            models.append(model)
            model_dates.append(dates[end-1])
        except Exception as exc:
            print(f"Stability fit failed at {dates[end-1]}: {exc}")

    rows = []
    for i in range(1, len(models)):
        if isinstance(models[i], PCAModel):
            rows.append({
                "date": model_dates[i],
                "subspace_distance": pca_subspace_distance(models[i-1], models[i]),
            })
        elif isinstance(models[i], SeparablePCAModel):
            d = separable_subspace_distance(models[i-1], models[i])
            rows.append({"date": model_dates[i], **d})

    return pd.DataFrame(rows).set_index("date") if rows else pd.DataFrame()


stability_df = rolling_stability_analysis(best_spec, vol_df, vega_vec, best_window, step=20)
display(stability_df.tail())

if len(stability_df):
    fig, ax = plt.subplots(figsize=(12, 4))
    stability_df.plot(ax=ax)
    ax.set_title(f"Rolling subspace stability: {best_model_name}")
    ax.set_ylabel("Subspace distance")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 16. Residual PCA: what is the selected model missing?

Run PCA on residual surface moves:

\[
R_t = \Delta \sigma_t - \widehat{\Delta \sigma}_t
\]

If the first residual component is structured and economically meaningful, the selected model is missing a real surface mode.

In [ ]:
residual_model = fit_pca_model(
    residual_dvol,
    transform="absolute",
    n_components=5,
    weights=np.ones_like(vega_vec),
    covariance_method="sample",
)

for j in range(min(5, residual_model.n_components)):
    loading = residual_model.component_loading_original_units(j).reshape(len(EXPIRIES), len(TENORS))
    heatmap_matrix(loading, EXPIRIES, TENORS, title=f"Residual PCA loading PC{j+1}")

print("Residual PCA eigenvalues:", residual_model.eigenvalues_)

## 17. Production-style helper

This function returns the latest fitted model, reduced factor exposures, component contributions and residual-risk table for a chosen spec.

In [ ]:
def fit_latest_and_report(
    spec: ModelSpec,
    vol_df: pd.DataFrame,
    vega_df: pd.DataFrame,
    window: int,
) -> Dict[str, Any]:
    vol_df = ensure_multiindex_columns(vol_df)
    expiries = list(vol_df.columns.get_level_values("expiry").unique())
    tenors = list(vol_df.columns.get_level_values("tenor").unique())
    vega_vec = align_vega_to_vol_columns(vega_df, vol_df.columns)
    vega_matrix = vega_df.loc[expiries, tenors].values

    move_data = compute_move_data(vol_df, transform=spec.transform)
    X = move_data["X"].values
    dvol = move_data["dvol"].values
    prev = move_data["prev"].values

    X_train = X[-window:]
    dvol_train = dvol[-window:]
    prev_train = prev[-window:]

    model = fit_model_from_spec(spec, X_train, vega_vec, len(expiries), len(tenors))
    X_hat = model.reconstruct_transformed(X_train)
    dvol_hat = inverse_transform_to_absolute(X_hat, prev_train, spec.transform)

    full_pnl = dvol_train @ vega_vec
    proj_pnl = dvol_hat @ vega_vec

    metrics = pnl_metrics(full_pnl, proj_pnl)

    if isinstance(model, PCAModel):
        contrib = pca_component_pnl_contributions(model, X_train, prev_train, vega_vec)
    else:
        contrib = separable_component_pnl_contributions(model, X_train, prev_train, vega_matrix)

    contrib.index = move_data["X"].index[-window:]

    residual_dvol = dvol_train - dvol_hat
    residual_std = residual_dvol.std(axis=0, ddof=1)
    residual_risk = np.abs(vega_vec) * residual_std

    residual_table = pd.DataFrame({
        "expiry": vol_df.columns.get_level_values("expiry"),
        "tenor": vol_df.columns.get_level_values("tenor"),
        "vega": vega_vec,
        "residual_vol_std": residual_std,
        "residual_risk": residual_risk,
    }).sort_values("residual_risk", ascending=False)

    return {
        "model": model,
        "metrics": metrics,
        "component_contributions": contrib,
        "residual_risk_table": residual_table,
        "full_pnl": pd.Series(full_pnl, index=move_data["X"].index[-window:]),
        "projected_pnl": pd.Series(proj_pnl, index=move_data["X"].index[-window:]),
    }


report = fit_latest_and_report(best_spec, vol_df, vega_df, best_window)
display(pd.Series(report["metrics"]))
display(report["residual_risk_table"].head(10))
display(report["component_contributions"].tail())

## 18. Suggested next extensions

Potential additions once the first real-data run works:

1. Add liquidity weights or mark-quality scores.
2. Compare PCA factors against actual liquid hedge instruments.
3. Add stressed-window validation.
4. Add regime labels, e.g. high rates vol / low rates vol / central-bank-event windows.
5. Add a greedy grid-point selection algorithm to choose real expiry/tenor projection points.
6. Add PnL PCA on bucket-level historical PnL contributions:
\[
Y_{t,i} = v_i \Delta \sigma_{t,i}
\]
7. Add daily export of:
   - factor exposures,
   - component PnL,
   - residual-risk heatmap,
   - unreliable buckets.